# Room classifier

In [ ]:
# Install + imports
!pip install -q datasets

import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from collections import Counter, defaultdict
from datasets import load_dataset
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             f1_score, confusion_matrix)

print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
# Settings — the only cell you edit
REPO_ID = "viethaa/concordia-hanoi-rooms"
LOCAL_DIR = None
HOLDOUT_SESSION = None

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 40
EPOCHS_FT = 25
UNFREEZE_LAYERS = 40
LR_HEAD = 1e-3
LR_FT = 1e-5
PATIENCE = 8
SEED = 1337

tf.keras.utils.set_random_seed(SEED)

TEAL, RUST = "#0D9488", "#C2410C"
INK, MUTED, GRID = "#1F2937", "#6B7280", "#E5E7EB"
plt.rcParams.update({
    "figure.dpi": 130, "font.size": 9, "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED, "text.color": INK, "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
})

### Data

In [ ]:
# Load the dataset and list what is in it, per class and per session
ds = (load_dataset("imagefolder", data_dir=LOCAL_DIR, split="train") if LOCAL_DIR
      else load_dataset(REPO_ID, split="train"))

# metadata.csv makes `label` a string, not an integer — build the index here
labels = ds["label"]
CLASSES = sorted(set(labels))
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

sessions = ds["session"] if "session" in ds.column_names else ["s1"] * len(ds)
SESSIONS = sorted(set(sessions))

grid = defaultdict(int)
for l, s in zip(labels, sessions):
    grid[(l, s)] += 1

print(pd.DataFrame(
    [[grid[(c, s)] for s in SESSIONS] + [sum(grid[(c, s)] for s in SESSIONS)]
     for c in CLASSES], index=CLASSES, columns=SESSIONS + ["total"]))

In [ ]:
# Split by capture session so the test set is a day the model never saw.
# With one session only, falls back to a random split and the score is inflated.
TRUSTWORTHY = len(SESSIONS) >= 2

if TRUSTWORTHY:
    holdout = HOLDOUT_SESSION or SESSIONS[-1]
    test_mask = np.array([s == holdout for s in sessions])
    SPLIT = f"held-out session {holdout}"
else:
    test_mask = np.random.default_rng(SEED).random(len(ds)) < 0.25
    SPLIT = "random split, single session — inflated"

print(SPLIT)


def to_arrays(indices):
    X = np.zeros((len(indices), *IMG_SIZE, 3), dtype=np.uint8)
    y = np.zeros(len(indices), dtype=np.int32)
    for n, i in enumerate(indices):
        row = ds[int(i)]
        X[n] = np.asarray(row["image"].convert("RGB").resize(IMG_SIZE))
        y[n] = CLASS_TO_IDX[row["label"]]
    return X, y


idx = np.arange(len(ds))
X_train, y_train = to_arrays(idx[~test_mask])
X_test, y_test = to_arrays(idx[test_mask])

print(f"train {len(X_train)}   test {len(X_test)}")
print(pd.Series(Counter(y_train))
      .rename(index={i: c for c, i in CLASS_TO_IDX.items()}).sort_index().to_string())

### Model

In [ ]:
# Frozen ImageNet backbone + a small new head. Augmentation hits training images only.
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.3, value_range=(0, 255)),
    layers.RandomContrast(0.3),
], name="augment")

backbone = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
backbone.trainable = False

inp = layers.Input(shape=(*IMG_SIZE, 3))
x = augment(inp)
x = layers.Lambda(preprocess_input)(x)
x = backbone(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(len(CLASSES), activation="softmax")(x)
model = Model(inp, out)

model.compile(optimizer=tf.keras.optimizers.Adam(LR_HEAD),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

cbs = lambda: [
    callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=PATIENCE // 2, min_lr=1e-7, verbose=1),
]

In [ ]:
# Round 1 — train the new head, backbone frozen
hist = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                 epochs=EPOCHS_HEAD, batch_size=BATCH_SIZE,
                 callbacks=cbs(), verbose=2)

In [ ]:
# Round 2 — unfreeze the top of the backbone, 100x lower learning rate.
# BatchNorm stays frozen: updating it on a small batch destabilises fine-tuning.
backbone.trainable = True
for layer in backbone.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(LR_FT),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

hist_ft = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                    epochs=EPOCHS_FT, batch_size=BATCH_SIZE,
                    callbacks=cbs(), verbose=2)

### Results

In [ ]:
# Accuracy and loss across both training rounds
acc = hist.history["accuracy"] + hist_ft.history["accuracy"]
val = hist.history["val_accuracy"] + hist_ft.history["val_accuracy"]
ls = hist.history["loss"] + hist_ft.history["loss"]
vls = hist.history["val_loss"] + hist_ft.history["val_loss"]
cut = len(hist.history["accuracy"])
ep = range(1, len(acc) + 1)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.3))

a1.plot(ep, acc, color=TEAL, lw=2, label="train")
a1.plot(ep, val, color=RUST, lw=2, label="held-out")
a1.set_ylim(0, 1.02); a1.set_ylabel("accuracy")

a2.plot(ep, ls, color=TEAL, lw=2, label="train")
a2.plot(ep, vls, color=RUST, lw=2, label="held-out")
a2.set_ylabel("loss")

for ax, title in ((a1, "Accuracy"), (a2, "Loss")):
    ax.axvline(cut + 0.5, color=GRID, lw=1.5, zorder=0)
    ax.set_xlabel("epoch")
    ax.yaxis.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)
    ax.legend(frameon=False)
    ax.set_title(title, color=INK, fontsize=10, loc="left", pad=10)

fig.text(0.5, -0.04, "vertical line = fine-tuning starts", ha="center",
         color=MUTED, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Metrics table -> metrics.csv
probs = model.predict(X_test, verbose=0)
pred = probs.argmax(axis=1)

p, r, f1, sup = precision_recall_fscore_support(
    y_test, pred, labels=range(len(CLASSES)), zero_division=0)

table = pd.DataFrame({
    "class": CLASSES,
    "precision": p.round(3),
    "recall": r.round(3),
    "f1": f1.round(3),
    "support": sup,
    "mean confidence": [round(float(probs[y_test == i].max(axis=1).mean()), 3)
                        if (y_test == i).any() else float("nan")
                        for i in range(len(CLASSES))],
})
table.loc[len(table)] = ["overall", round(float(p.mean()), 3), round(float(r.mean()), 3),
                         round(float(f1_score(y_test, pred, average="macro",
                                              zero_division=0)), 3),
                         int(sup.sum()), round(float(probs.max(axis=1).mean()), 3)]

n, k = len(y_test), int((pred == y_test).sum())
print(f"accuracy {accuracy_score(y_test, pred):.1%}  ({k}/{n})  [{SPLIT}]")

table.to_csv("metrics.csv", index=False)
display(table.style.hide(axis="index"))

# which rooms get mistaken for which
cm = confusion_matrix(y_test, pred, labels=range(len(CLASSES)))
mix = [(CLASSES[i], CLASSES[j], int(cm[i, j])) for i in range(len(CLASSES))
       for j in range(len(CLASSES)) if i != j and cm[i, j]]
for a, b, c in sorted(mix, key=lambda t: -t[2]):
    print(f"{a} -> {b}: {c}")

### Save

In [ ]:
model.save("room_classifier.keras")
with open("classes.txt", "w") as fh:
    fh.write("\n".join(CLASSES))
print(CLASSES)